# 01 Von der Frage zur strukturierten Antwort

**Reihe:** LangChain durch Literate Programming lernen, Notebook 1 von 4.

**Leitfrage:** Wie kann ein Modul-Assistent eine Frage anhand eines kurzen Textes beantworten
und dabei eine Quelle angeben?

Dieses Notebook beginnt mit einer Frage zur mitgelieferten Bitcoin-Unterlage.
Schritt für Schritt entsteht daraus eine Verarbeitungskette: Die Anfrage wird vorbereitet,
an das Modell übergeben und die Antwort in eine feste Form gebracht.

Am Ende enthält das Ergebnis zwei Felder: **Antwort** und **Quelle**.
Der Text wird zunächst aus der ersten Seite dieser PDF gelesen. Ein Gesprächsverlauf folgt in Notebook 02,
die automatische Suche in PDF-Unterlagen in Notebook 03.

LangChain stellt die Bausteine bereit, mit denen sich diese Schritte verbinden lassen.
Die folgenden Abschnitte führen sie einzeln ein.

### Orientierung für den Einstieg

Die Aufgabenstellung verlangt die Lernkonzepte, Erklärungen, Fehlerbeispiele, den Promptvergleich und eine eigene Übung. Demo-Modus, zusätzliche JSON-Prüfungen, CSV-Export und die besonders kleinen Codezellen sind didaktische Ergänzungen dieses Projekts. Der separate Lernleitfaden führt zuerst durch den Kernweg und erklärt anschliessend diese Zusätze.


## Lernziele

Nach dem Durcharbeiten können Leserinnen und Leser:

- die Rollen von System-, Human- und AI-Nachrichten unterscheiden und erklären, warum eine neue Anfrage den bisherigen Verlauf nicht automatisch kennt;
- einen Prompt mit Platzhaltern erstellen und Prompt, Modell und Parser zu einer Chain verbinden;
- eine vollständige Antwort mit einer schrittweise übertragenen Streaming-Antwort vergleichen;
- gemeldete Tokenzahlen einordnen und eine Kostenrechnung mit ausdrücklich angenommenen Preisen nachvollziehen.

## Setup

Die Zellen werden von oben nach unten ausgeführt. Dabei bleiben Variablen im laufenden
Python-Kernel verfügbar. Jede folgende Codezelle hat zunächst nur eine kleine Aufgabe.

Zuerst werden Python-Hilfsmittel geladen: `os` liest Einstellungen, `sys` liefert die
Python-Version und `Path` bildet Dateipfade. `version` liest Paketversionen aus.
`load_dotenv` lädt später die lokale Datei `.env`. Es wird noch kein Modell aufgerufen.

In [1]:
import os
import sys
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

Die Versionsprüfung verhindert einen Start mit einer nicht unterstützten Python-Version.
Die beiden Ausgaben helfen später bei der Suche nach Installationsproblemen.

In [2]:
assert sys.version_info[:2] in ((3, 12), (3, 13)), 'Python 3.12 oder 3.13 erforderlich'
print('Python:', sys.version.split()[0])
print('LangChain Core:', version('langchain-core'))

Python: 3.13.15
LangChain Core: 1.7.3


Die Projektdateien werden über relative Pfade gefunden. Jupyter kann im Projektordner
oder direkt im Ordner `notebooks` gestartet sein. Im zweiten Fall führt `.parent`
eine Ebene nach oben. Ein persönlicher Benutzerpfad ist dadurch nicht nötig.

In [3]:
projektordner = Path.cwd()
if projektordner.name == 'notebooks':
    projektordner = projektordner.parent

`.env` legt die Betriebsart fest. `demo` verwendet vorbereitete Antworten ohne API-Aufruf.
`bfh` sendet echte Modellanfragen mit dem lokal gespeicherten Schlüssel.
Fehlt die Einstellung, wird `demo` verwendet. Der Schlüssel wird nicht ausgegeben.

In [4]:
load_dotenv(projektordner / '.env', override=False)
modus = os.getenv('LLM_MODE', 'demo').lower()
assert modus in ('demo', 'bfh'), 'LLM_MODE muss demo oder bfh sein.'
print('Betriebsart:', modus)

Betriebsart: bfh


## Konzept Erklärung

Ein **Sprachmodell** erzeugt eine Antwort auf eine Anfrage. Damit sich die Antwort auf eine
bestimmte Unterlage beziehen kann, erhält das Modell neben der Frage einen passenden Text.
Dieser mitgegebene Text heisst hier **Kontext**. Eine Anweisung beschreibt, wie das Modell
Frage und Kontext verwenden soll. Diese Anweisung ist Teil des **Prompts**.

LangChain übergibt solche Informationen als Nachrichten mit unterschiedlichen Rollen:

| Rolle | Aufgabe im Beispiel |
|---|---|
| System | Gibt die Regeln vor, etwa nur den bereitgestellten Text zu verwenden. |
| Human | Enthält den Text und die Frage. |
| AI | Enthält die zurückgegebene Modellantwort. |

Eine neue Anfrage enthält zunächst nur die Nachrichten, die ihr ausdrücklich mitgegeben werden.
Frühere Fragen und Antworten müssen erneut übergeben werden, wenn sie für die neue Antwort
wichtig sind. Darauf baut das Gesprächsgedächtnis in Notebook 02 auf.

Der erste Versuch hat damit einen überschaubaren Ablauf:

**Text und Frage → Nachrichten → Modell → Antwort**

## Experiment

### 1 Die Bitcoin Unterlage finden

Die erste Frage bezieht sich auf `Eigenschaften-Bitcoin.pdf`. Diese einseitige
Modulunterlage liegt im Repository unter `data/modulunterlagen`. Die zweite PDF
behandelt Open Banking und kommt später bei der gemeinsamen Dokumentensuche hinzu.

Der Pfad beginnt beim Projektordner. Die Prüfung mit `is_file()` zeigt frühzeitig,
ob die Datei am erwarteten Ort liegt. Ausgegeben wird nur ihr Dateiname.

In [5]:
pdf_datei = projektordner / 'data' / 'modulunterlagen' / 'Eigenschaften-Bitcoin.pdf'
assert pdf_datei.is_file(), 'Die Bitcoin-Unterlage fehlt in data/modulunterlagen.'
print('Unterlage:', pdf_datei.name)

Unterlage: Eigenschaften-Bitcoin.pdf


### 2 Den Text der ersten Seite auslesen

`PdfReader` aus dem Paket `pypdf` öffnet die PDF. `pages[0]` bedeutet die erste Seite,
weil Python bei null zu zählen beginnt. `extract_text()` liest ihren Text aus.
Dieser Text wird in `kontext` gespeichert und zur Kontrolle angezeigt.

Hier wird noch keine passende Textstelle gesucht. Die ganze erste Seite wird als
bekannte Informationsbasis verwendet. Das spätere RAG-Notebook ergänzt die Suche.

In [6]:
from pypdf import PdfReader

pdf = PdfReader(pdf_datei)
kontext = pdf.pages[0].extract_text()
assert kontext and kontext.strip(), 'Aus der PDF wurde kein Text gelesen.'
print(kontext)

Eigenschaften Bitcoin 
Meisser, Luzius, Hauser-Spühler, Gabriel 
https://doi.org/10.5167/uzh-153361 
Allgemeine In- 
formationen zu 
Bitcoin 
• Bitcoin gehört zu den 20 grössten Währungen 
=>gemessen an Geldmenge 
• Die Kryptowährung «Bitcoin» ist auf der «Blockchain» aufgebaut. 
• Bitcoins fallen ebenfalls in die Konkursmasse bei einer Liquidie- 
rung eines Unternehmens oder einer Privatperson. 
Was ist Bitcoin? • Bitcoin ist eine dezentral organisierte Internetwährung 
• Der Wechselkurs entsteht wie bei anderen Währungen durch An- 
gebot und Nachfrage auf dem freien Markt. 
• Der Wert eines Bitcoins wird von niemandem garantiert. 
• Die Menge an Bitcoins ist vordefiniert und auf 21 Millionen Bitcoins 
begrenzt. 
• Anteile an Bitcoins oder ganze Bitcoins sind immer klar einer Ad- 
resse zugeordnet.  
Informationen 
zu Transaktio- 
nen von Bit- 
coins 
• Transaktionen finden im Bitcoin-System direkt von Person zu Per- 
son statt. 
• Der zwischengeschalteter Zahlungsdienstleister (Bank)

### 3 Die Frage und die Quelle festlegen

Die Frage lautet, auf welche Gesamtmenge Bitcoin laut der Unterlage begrenzt ist.
Die Quelle besteht aus dem Dateinamen und der lesbaren Seitennummer 1.
Text, Frage und Quelle bleiben getrennte Variablen, damit sie einzeln austauschbar sind.

In [7]:
frage = 'Auf welche Gesamtmenge ist Bitcoin laut der Unterlage begrenzt?'
quelle = f'{pdf_datei.name}, S. 1'
print('Frage:', frage)
print('Quelle:', quelle)

Frage: Auf welche Gesamtmenge ist Bitcoin laut der Unterlage begrenzt?
Quelle: Eigenschaften-Bitcoin.pdf, S. 1


### 4 Die Modellschnittstelle vorbereiten

LangChain verwendet ein Modellobjekt, an das später die Nachrichten übergeben werden.
`ChatOpenAI` spricht im BFH-Modus die OpenAI-kompatible BFH-Schnittstelle an.
`FakeListChatModel` liefert im Demo-Modus einen vorbereiteten Text.
Zunächst werden nur die beiden Klassen importiert.

Dokumentation: [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)
und [BFH Inference](https://infra.pages.ti.bfh.ch/mlmp/src/llm/).

In [8]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_openai import ChatOpenAI

Für den Demo-Modus wird eine passende Beispielantwort hinterlegt. Sie stammt aus
dem bekannten Bitcoin-Text, wird aber nicht von einem Sprachmodell erzeugt.
Deshalb eignet sie sich zur Prüfung des Ablaufs und nicht zur Bewertung eines Modells.

In [9]:
demo_antwort = 'Laut der Unterlage ist die Gesamtmenge auf 21 Millionen Bitcoins begrenzt.'
modell = FakeListChatModel(responses=[demo_antwort])

Für den BFH-Modus werden Modellname, Adresse und Schlüssel aus den Einstellungen
gelesen. Die Adresse hat einen Vorgabewert, falls `.env` sie nicht festlegt.
Es wird weiterhin keine Anfrage gesendet und kein Schlüssel ausgegeben.

In [10]:
bfh_modell = os.getenv('BFH_CHAT_MODEL', 'gpt-oss:120b')
bfh_adresse = os.getenv('BFH_BASE_URL', 'https://inference.proxy.ti.bfh.ch/v1')
bfh_schluessel = os.getenv('BFH_API_KEY')

### BFH Adresse und Modell kontrollieren

Die Adresse bestimmt, an welchen Dienst und welchen API-Pfad die Anfrage geht.
Der Modellname bestimmt, welches Modell dieser Dienst verwenden soll. Die beiden
Ausgaben unten zeigen diese Einstellungen, ohne den Schlüssel auszugeben.

Für den BFH-Proxy lautet die hier geprüfte API-Basisadresse
`https://inference.proxy.ti.bfh.ch/v1`. Bei diesem Proxy führt `/api/v1` zu HTTP 404.
Die Bibliotheksmeldung `OpenAIModelNotFoundError` kann auch bei einem falschen
API-Pfad erscheinen; sie beweist allein kein fehlendes Modell.


In [11]:
print('BFH-API-Adresse:', bfh_adresse)
print('BFH-Modell:', bfh_modell)

BFH-API-Adresse: https://inference.proxy.ti.bfh.ch/v1
BFH-Modell: qwen3.8-27b


Nur der BFH-Modus benötigt einen persönlichen Schlüssel. Die nächste Prüfung erklärt
einen fehlenden Schlüssel vor dem Aufbau der Verbindung. Im Demo-Modus geschieht nichts.

In [12]:
if modus == 'bfh' and not bfh_schluessel:
    raise ValueError('BFH_API_KEY fehlt in der lokalen .env-Datei.')

Im BFH-Modus ersetzt `ChatOpenAI` das Demo-Objekt. Die Parameter nennen Modell und
Verbindung. `temperature=0` verringert die Zufälligkeit, garantiert aber keine identischen
Antworten. `timeout` und `max_retries` begrenzen Wartezeit und Wiederholungsversuche.
Die letzten beiden Einstellungen verwenden die Chat-Completions-Schnittstelle ohne
zusätzliche Streaming-Verbrauchsdaten. Im Demo-Modus bleibt das vorbereitete Objekt bestehen.

In [13]:
if modus == 'bfh':
    modell = ChatOpenAI(
        model=bfh_modell,
        base_url=bfh_adresse,
        api_key=bfh_schluessel,
        temperature=0,
        timeout=90,
        max_retries=1,
        use_responses_api=False,
        stream_usage=False,
    )

### 5 Eine erste Anfrage senden

Die Systemnachricht legt die Antwortregel fest. Die Human-Nachricht enthält den Text und die
Frage. `invoke()` führt einen vollständigen Aufruf aus. Die Antwort kommt als `AIMessage`
zurück; ihr Feld `content` enthält den Text.

Der Weg lässt sich nun am Code verfolgen: Python erstellt Nachrichten, LangChain übergibt
sie an die Modellschnittstelle, diese ruft im BFH-Modus die API auf und das Ergebnis kommt
als Nachricht zurück. Im Demo-Modus wird der API-Schritt durch den vorbereiteten Text ersetzt.

In [14]:
from langchain_core.messages import SystemMessage, HumanMessage

nachrichten = [
    SystemMessage(content='Antworte auf Deutsch und nur anhand des Textes. Fehlt die Antwort, sage dies ausdrücklich.'),
    HumanMessage(content=f'Text: {kontext}\nFrage: {frage}'),
]

erste_antwort = modell.invoke(nachrichten)
print('Antworttyp:', type(erste_antwort).__name__)
print('Antwort:', erste_antwort.content)

Antworttyp: AIMessage
Antwort: Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf **21 Millionen Bitcoins** begrenzt.


### 6 Was enthält eine Anschlussfrage?

Die Anschlussfrage «Und wie entsteht der Wechselkurs?» benötigt den vorherigen
Bezug. Ohne den Verlauf sieht die neue Anfrage nur diesen einen Satz. Mit Verlauf enthält sie
auch den Ausgangstext und die erste Antwort.

Die folgende Zelle zeigt beide Nachrichtenlisten. Es wird keine weitere Modellantwort benötigt:
Entscheidend ist zunächst, welche Informationen überhaupt ankommen. Im Demo-Modus wäre eine
passend klingende Antwort kein Nachweis, weil sie unabhängig von der Frage vorgegeben ist.

In [15]:
anschlussfrage = HumanMessage(content='Und wie entsteht der Wechselkurs?')

ohne_verlauf = [anschlussfrage]
mit_verlauf = nachrichten + [erste_antwort, anschlussfrage]

print('Ohne Verlauf:', [(n.type, n.content) for n in ohne_verlauf])
print('Mit Verlauf:', [n.type for n in mit_verlauf])
assert len(ohne_verlauf) == 1
assert len(mit_verlauf) == 4

Ohne Verlauf: [('human', 'Und wie entsteht der Wechselkurs?')]
Mit Verlauf: ['system', 'human', 'ai', 'human']


### 7 Die Antwort schrittweise anzeigen

`invoke()` liefert die fertige Antwort. **Streaming** stellt dagegen bereits einzelne Teile
bereit, während die Antwort entsteht. `stream()` liefert diese Teile nacheinander.
Die Schleife gibt jeden Textteil unmittelbar aus.

Auch hier gilt: Im Demo-Modus wird der vorbereitete Text stückweise ausgegeben.
Eine Aussage über die Geschwindigkeit des BFH-Modells ist damit nicht möglich.

In [16]:
for teil in modell.stream(nachrichten):
    print(teil.content, end='', flush=True)
print()



Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf **21 Millionen Bitcoins** begrenzt.


### 8 Tokenverbrauch und Kosten einordnen

Ein **Token** ist ein Textbaustein, den ein Modell verarbeitet. Ein Token entspricht nicht
immer einem ganzen Wort. Zur Eingabe zählen auch die Anweisungen und der mitgegebene Kontext.

Wenn der Dienst Nutzungsdaten zurückgibt, enthält `usage_metadata` die gemeldeten Zahlen für
Ein- und Ausgabe. Fehlen sie, kann daraus kein tatsächlicher Verbrauch abgeleitet werden.
Der Demo-Modus meldet keine echten Modell-Tokens.

In [17]:
nutzung = erste_antwort.usage_metadata

if nutzung:
    print('Eingabe-Tokens:', nutzung['input_tokens'])
    print('Ausgabe-Tokens:', nutzung['output_tokens'])
else:
    print('Keine Tokenzahlen vom Modell gemeldet.')

Eingabe-Tokens: 579
Ausgabe-Tokens: 120


Eine Kostenabschätzung braucht zusätzlich die Preise des jeweiligen Dienstes. Die Formel lautet:

**Kosten = Eingabe-Tokens × Eingabepreis + Ausgabe-Tokens × Ausgabepreis**

Werden Preise pro Million Tokens angegeben, muss das Ergebnis durch eine Million geteilt werden.
Die folgende Rechnung verwendet ausdrücklich angenommene Werte: 1 000 Eingabe-Tokens,
200 Ausgabe-Tokens sowie 0.20 und 0.80 CHF pro Million Tokens. Das sind weder gemessene
Tokenzahlen noch BFH-Tarife. Falls echte Nutzungsdaten vorliegen, wird zusätzlich damit gerechnet,
weiterhin mit denselben angenommenen Preisen.

In [18]:
def kosten(input_tokens, output_tokens, preis_input, preis_output):
    return (input_tokens * preis_input + output_tokens * preis_output) / 1_000_000

print('Rechenbeispiel:', kosten(1000, 200, 0.20, 0.80), 'CHF')
assert abs(kosten(1000, 200, 0.20, 0.80) - 0.00036) < 1e-12

if nutzung:
    schaetzung = kosten(nutzung['input_tokens'], nutzung['output_tokens'], 0.20, 0.80)
    print('Gemeldete Tokenzahlen mit angenommenen Preisen:', schaetzung, 'CHF')

Rechenbeispiel: 0.00036 CHF
Gemeldete Tokenzahlen mit angenommenen Preisen: 0.0002118 CHF


### 9 Die gewünschte Antwortform festlegen

Die bisherige Antwort ist freier Text. Für die weitere Verarbeitung soll sie zwei fest
benannte Felder enthalten: `antwort` und `quelle`. Beide Felder sollen Text enthalten.
Ein solches **Schema** beschreibt die erwartete Form der Daten.

Die kleine Klasse `Antwort` legt dieses Schema mit Pydantic fest. Ein **OutputParser** liest
anschliessend den Modelltext und prüft ihn gegen das Schema. Dadurch wird beispielsweise
erkannt, wenn ein Feld fehlt. Ob die Antwort fachlich richtig ist, bleibt eine eigene Prüfung.

In [19]:
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser

class Antwort(BaseModel):
    antwort: str
    quelle: str

parser = PydanticOutputParser(pydantic_object=Antwort)
print('Erwartete Felder:', list(Antwort.model_fields))

Erwartete Felder: ['antwort', 'quelle']


### 10 Eine wiederverwendbare Promptvorlage erstellen

Eine **Promptvorlage** enthält Platzhalter statt fest eingebauter Inhalte.
`{kontext}`, `{quelle}` und `{frage}` werden bei einem Aufruf durch die jeweiligen Werte ersetzt.
Die Vorlage bleibt dadurch für andere Textauszüge und Fragen verwendbar.

`ChatPromptTemplate` erstellt daraus die Nachrichten. Zusätzlich beschreibt
`parser.get_format_instructions()`, wie die Antwort aufgebaut sein muss. Diese gleichbleibende
Formatvorgabe wird mit `partial()` bereits in die Vorlage eingesetzt. Beim späteren Aufruf
müssen nur noch Text, Quelle und Frage übergeben werden.

Dokumentation: [ChatPromptTemplate](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate).

In [20]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Antworte auf Deutsch und nur anhand des Textes. '
     'Fehlt die Information, sage dies ausdrücklich und verwende als Quelle keine Quelle. '
     'Ansonsten übernimm die mitgegebene Quellenangabe. {formatvorgabe}'),
    ('human', 'Text: {kontext}\nQuelle: {quelle}\nFrage: {frage}'),
]).partial(formatvorgabe=parser.get_format_instructions())

eingaben = {'kontext': kontext, 'quelle': quelle, 'frage': frage}

Vor einer weiteren Modellanfrage lohnt sich ein Blick auf die ausgefüllte Vorlage.
`prompt.invoke(eingaben)` setzt die Werte ein; `to_string()` macht die entstandenen Nachrichten
lesbar. Dieser Schritt ruft noch kein Modell auf. Fehlende oder falsch zugeordnete Werte lassen
sich dadurch früh erkennen.

In [21]:
vorbereiteter_prompt = prompt.invoke(eingaben)
print(vorbereiteter_prompt.to_string())

System: Antworte auf Deutsch und nur anhand des Textes. Fehlt die Information, sage dies ausdrücklich und verwende als Quelle keine Quelle. Ansonsten übernimm die mitgegebene Quellenangabe. The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"antwort": {"title": "Antwort", "type": "string"}, "quelle": {"title": "Quelle", "type": "string"}}, "required": ["antwort", "quelle"]}
```
Human: Text: Eigenschaften Bitcoin 
Meisser, Luzius, Hauser-Spühler, Gabriel 
https://doi.org/10.5167/uzh-153361 
Allgemeine In- 
formationen zu 
Bitcoin 
• Bitcoin gehört zu den 20 grössten Währungen 
=

### 11 Prompt Modell und Parser verbinden

Eine **Chain** verbindet Verarbeitungsschritte. In diesem Beispiel gibt es drei:

**Promptvorlage → Modell → OutputParser**

LangChain nennt diese Schreibweise **LCEL**, kurz für LangChain Expression Language.
Der Operator `|` übergibt das Ergebnis eines Schrittes an den nächsten. Die Vorlage liefert
Nachrichten, das Modell liefert eine Antwortnachricht und der Parser liefert die geprüfte
Antwort mit den beiden Feldern.

Im Demo-Modus ist für diesen Versuch ein vorbereiteter JSON-Text nötig, weil der erste
Demonstrationswert freier Text war. JSON ist ein Textformat für strukturierte Daten.
`json.dumps()` erzeugt den passenden Text aus einem Python-Dictionary. Im BFH-Modus wird
weiterhin das echte Modell verwendet, das die Formatvorgabe selbst umsetzen muss.

In [22]:
import json

json_modell = modell
if modus == 'demo':
    demo_json = json.dumps({'antwort': demo_antwort, 'quelle': quelle}, ensure_ascii=False)
    json_modell = FakeListChatModel(responses=[demo_json])

chain = prompt | json_modell | parser
ergebnis = chain.invoke(eingaben)

print('Antwort:', ergebnis.antwort)
print('Quelle:', ergebnis.quelle)
assert isinstance(ergebnis, Antwort)

Antwort: Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf 21 Millionen Bitcoins begrenzt.
Quelle: Eigenschaften-Bitcoin.pdf, S. 1


### 12 Den Datenfluss Schritt für Schritt untersuchen

Die kurze Schreibweise mit `|` fasst mehrere Schritte zusammen. Zur Fehlersuche lassen sie
sich einzeln ausführen. Die folgende Zelle zeigt die drei Zwischenergebnisse.

Für einen Debugger-Durchlauf kann ein Haltepunkt bei `nachrichtenwert` gesetzt werden.
Beim schrittweisen Ausführen lässt sich verfolgen, wie aus dem Dictionary `eingaben` zuerst
Nachrichten, danach Modelltext und schliesslich eine geprüfte `Antwort` werden. So ist bei einem
Fehler erkennbar, in welchem Schritt er entsteht.

In [23]:
nachrichtenwert = prompt.invoke(eingaben)
modellantwort = json_modell.invoke(nachrichtenwert)
gepruefte_antwort = parser.invoke(modellantwort)

print('Nach der Vorlage:', type(nachrichtenwert).__name__)
print('Nach dem Modell:', modellantwort.content)
print('Nach dem Parser:', gepruefte_antwort.model_dump())

Nach der Vorlage: ChatPromptValue
Nach dem Modell: {"antwort": "Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf 21 Millionen Bitcoins begrenzt.", "quelle": "Eigenschaften-Bitcoin.pdf, S. 1"}
Nach dem Parser: {'antwort': 'Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf 21 Millionen Bitcoins begrenzt.', 'quelle': 'Eigenschaften-Bitcoin.pdf, S. 1'}


## Häufige Fehler

### Eine benötigte Promptvariable fehlt

Die Vorlage benötigt `kontext`, `quelle` und `frage`. Die nächste Zelle übergibt absichtlich
nur die Frage. Dadurch fehlen zwei Werte. Bereits das Ausfüllen der Vorlage bricht mit einem
`KeyError` ab; es wird keine Modellanfrage gesendet.

`try` und `except` fangen hier ausschliesslich den erwarteten Fehlertyp ab, damit der geplante
Fehler sichtbar wird und das Notebook trotzdem weiterläuft. Die Fehlermeldung nennt die
fehlenden Variablen.

In [24]:
try:
    prompt.invoke({'frage': frage})
except KeyError as fehler:
    print('Erwarteter Fehler:', type(fehler).__name__)
    print(str(fehler)[:450])
else:
    raise AssertionError('Der erwartete Fehler ist nicht aufgetreten.')

Erwarteter Fehler: KeyError
"Input to ChatPromptTemplate is missing variables {'quelle', 'kontext'}.  Expected: ['frage', 'kontext', 'quelle'] Received: ['frage']\nNote: if you intended {quelle} to be part of the string and not a variable, please escape it with double curly braces like: '{{quelle}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "


### Die Eingaben vervollständigen

Die Korrektur besteht darin, alle drei Werte zu übergeben. Das bereits definierte Dictionary
`eingaben` enthält sie. Die folgende Zelle prüft nur die Vorbereitung des Prompts und verursacht
deshalb auch im BFH-Modus keine weitere Modellanfrage.

In [25]:
korrigierter_prompt = prompt.invoke(eingaben)
print('Die Vorlage wurde vollständig ausgefüllt.')
assert len(korrigierter_prompt.to_messages()) == 2

Die Vorlage wurde vollständig ausgefüllt.


## Prompt Logbuch

### Was gehört hier hinein?

Ein **Prompt** ist der vollständige Arbeitsauftrag an das Modell: Anweisung, Frage und mitgegebene Informationen. Die Modellantwort ist das Ergebnis, nicht der Prompt. Im Code können diese Bestandteile auf System- und Human-Nachricht verteilt sein.

Die Aufgabenstellung verlangt zuerst einen selbstständig geschriebenen Prompt und danach eine mit KI überarbeitete Fassung. **Die folgenden Texte wurden auf Wunsch als leicht verständliche KI-Vorschläge vorbereitet. Sie sind keine nachträglich erfundenen eigenen Erstprompts und noch kein ausgeführter Vergleich.** „Final“ bedeutet hier die zuletzt verwendete Fassung, nicht eine perfekte oder besonders lange Anweisung.

### Formulierungshilfe zu diesem Notebook

**Thema: Eine erste Antwort mit Quelle**

**Vorschlag A – einfacher Ausgangspunkt, von KI formuliert**

```text
Erkläre mir anhand des Textes, auf welche Gesamtmenge Bitcoin begrenzt ist.
Text: Die Menge an Bitcoins ist auf 21 Millionen Bitcoins begrenzt.
Quelle: Eigenschaften-Bitcoin.pdf, S. 1
```

**Vorschlag B – genauerer Arbeitsauftrag, von KI formuliert**

```text
Beantworte die Frage auf Deutsch und nur mit Informationen aus dem Text.
Gib JSON mit den Feldern "antwort" und "quelle" zurück.
Übernimm die angegebene Quelle. Fehlt die Information, sage das und verwende "keine Quelle".
Frage: Auf welche Gesamtmenge ist Bitcoin begrenzt?
Text: Die Menge an Bitcoins ist auf 21 Millionen Bitcoins begrenzt.
Quelle: Eigenschaften-Bitcoin.pdf, S. 1
```

**Was wird genauer?** Die erste Fassung stellt die Frage und liefert den Text. Die Überarbeitung legt zusätzlich Antwortformat, Quelle und den Umgang mit fehlenden Informationen fest. Ein JSON-Parser kann das Format prüfen, aber nicht beweisen, dass die Antwort stimmt.

### Eigener Nachweis nach dem tatsächlichen Versuch

Die Formulierungshilfen können zum Üben verwendet werden. Der geforderte eigene Erstprompt muss für einen neuen Versuch ohne vorherige KI-Überarbeitung entstehen.

- **Mein erster vollständiger Prompt:** beim eigenen Versuch selbst eintragen.
- **Meine tatsächlich verwendete Überarbeitung:** nach dem Versuch eintragen.
- **Was änderte sich an den Antworten?** Beide tatsächlichen Antworten, Datum und Modell festhalten. Auch keine Verbesserung oder eine neue falsche Aussage sind gültige Beobachtungen.

Die von Codex ausgeführten technischen Prüfungen sind im Prüfprotokoll dokumentiert. Sie werden nicht als persönlicher Promptvergleich ausgegeben.


## Zusammenfassung

Aus einer einzelnen Frage ist eine kleine Verarbeitungskette entstanden. Der Text und die Frage
werden als Nachrichten übergeben. Eine Promptvorlage hält die wiederverwendbaren Anweisungen fest.
Modell und Parser liefern daraus eine Antwort mit Quellenangabe.

Die bisherigen Schritte beantworten eine einzelne Anfrage. Ein Gesprächsverlauf wird noch nicht
automatisch gespeichert. Notebook 02 ergänzt diese Fähigkeit und vergleicht Möglichkeiten,
frühere Nachrichten bei einer Anschlussfrage erneut zu berücksichtigen.

### Lernziele überprüfen

- Welche Angaben stehen in der Systemnachricht und welche in der Human-Nachricht? Was fehlt einer Anschlussfrage ohne Verlauf?
- Welche drei Schritte verbindet die Chain, und welche Art von Ergebnis gibt jeder Schritt weiter?
- Was unterscheidet `invoke()` von `stream()`?
- Welche Tokenzahlen wurden tatsächlich gemeldet? Welche Werte der Kostenrechnung sind lediglich Annahmen?

## Übungsaufgabe

Zur Bitcoin-Unterlage soll eine weitere Frage formuliert werden, die der Text
beantworten kann, und eine Frage, für die eine Information fehlt. Für eine der beiden Fragen
ist zunächst ein eigener Prompt zu schreiben. Danach kann eine Überarbeitung mit KI-Hilfe
erfolgen. Der Vergleich wird im Prompt-Logbuch festgehalten.

Die inhaltliche Prüfung erfolgt mit dem BFH-Modell: Stützt sich die Antwort auf den Text,
stimmt die Quelle und wird fehlende Information benannt? Im Demo-Modus bleiben die Antworten
fest vorgegeben; eine geänderte Frage führt deshalb nicht automatisch zu einer passenden Antwort.

Als Frage ohne Beleg eignet sich beispielsweise der heutige Bitcoin-Kurs; dieser Wert
steht nicht in der Unterlage.

Die folgende Codezelle bleibt für die eigene Lösung leer.